# 01 Playground — JEV-JA を手元で動かす

日常の試行はこのノートで行う。state / questions / モデルをセルで変えて再実行するだけ。

`config/models.yaml` のキーでモデルを選ぶ。学習済みヘッド（`checkpoints/<name>`）があれば自動で使われる。

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
os.environ['JEV_LAB_CONFIG'] = os.path.abspath('../config/models.yaml')
from jev_lab.adapters import get_adapter, list_models
from jev_lab.schema import ChoiceQ, ScoreQ, NoulQ
[m['name'] for m in list_models()]

In [ ]:
ad = get_adapter('jev_ja_30m'); ad.load(); ad.info()

## 入力を変えて試す

In [ ]:
state = '津市栗真町屋町の自宅2階にいます。道路が30cmくらい冠水し、雨はまだ強い。足首を捻挫していて走れません。'
context = ['栗真町屋町周辺は志登茂川の洪水浸水想定区域（想定最大 0.5〜3.0m）']
questions = [
    ChoiceQ(id='hazard', instructions='この状況で主に起きている災害は何か', options=['洪水・内水氾濫','津波','土砂災害','地震の建物被害','不明']),
    ScoreQ(id='urgency', instructions='今すぐ移動を始める必要性', levels=['低い','やや低い','中程度','高い','非常に高い']),
    NoulQ(id='vertical', instructions='屋外への水平避難より、その場で上階にとどまる垂直避難が適切か'),
]
res = ad.decide(state, questions, context)
print(f'{res.latency_ms:.0f} ms')
for a in res.answers:
    print(a.id, getattr(a, 'choice', None) or getattr(a, 'level', None) or a.noul, [round(p, 3) for p in a.distribution.probabilities])

## 分布を描く

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(res.answers), figsize=(4*len(res.answers), 3))
for ax, a in zip(axes, res.answers):
    ax.bar(a.distribution.labels, a.distribution.probabilities); ax.set_title(a.id); ax.tick_params(axis='x', rotation=45)
plt.tight_layout()

## 複数モデルを並べる

（`pip install gliner2` 等が必要なものは、依存を入れてから）

In [ ]:
for name in ['jev_ja_30m', 'nli_mdeberta']:
    try:
        r = get_adapter(name).decide(state, questions, context)
        print(name, r.probability_kind, f'{r.latency_ms:.0f}ms', [(a.id, round(a.distribution.probabilities[a.distribution.argmax], 2)) for a in r.answers])
    except Exception as e:
        print(name, 'skip:', e)

## 文脈（RAG）あり／なし の違い

In [ ]:
r0 = ad.decide(state, questions, None); r1 = ad.decide(state, questions, context)
for a0, a1 in zip(r0.answers, r1.answers):
    print(a0.id, 'なし', [round(p,2) for p in a0.distribution.probabilities], ' あり', [round(p,2) for p in a1.distribution.probabilities])

## 次: 学習 → ONNX → 公開

```
python -m jev_lab.train.train_jev_ja --train data/train.jsonl --val data/val.jsonl --out checkpoints/jev_ja_30m --epochs 2 --augment 0.7
python -m jev_lab.export.export_onnx --model jev_ja_30m
python -m jev_lab.bench.run --data bench/data/sample_ja.jsonl --models jev_ja_30m --publish
git add -A && git commit -m 'update model' && git push
```